In [1]:
# Standard library
import json
import os
import re
import warnings
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
from matplotlib import colormaps
from matplotlib.colors import Normalize
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

# Show all rows and prevent column truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)  # Auto-detect width

In [2]:
metric_groups = {
    "Feature_Relevance": [
        "anova_f_mean", 
        "mutual_info_mean"
    ],
    "Local_Overlap": [
        "pca_centroid_distance_pca_centroid_score", 
        "mahalanobis_class_distance_mean"
    ],
    "Boundary_Hardness": [
        "svm_margin_mean", 
        "class_proba_entropy_mean"
    ],
    "Global_Structure": [
        "intrinsic_dimensionality_intrinsic_dimensionality_percent", 
        "calinski_harabasz_calinski_harabasz_score"
    ],
    "Class_Distribution_Separation": [
        "class_confusion_entropy_confusion_entropy", 
        "class_imbalance_normalized_entropy"
    ]
}

In [3]:
# Invert selected metrics (where higher = easier)
metrics_to_invert = {
    "anova_f_mean",
    "mutual_info_mean",
    "svm_margin_mean",
    "pca_centroid_score",
    "mahalanobis_class_distance_mean",
    "calinski_harabasz_calinski_harabasz_score",
    "class_imbalance_normalized_entropy"
}

In [4]:
row_order = [
    "CIC_IDS_2017",
    "CIC_IOT_Dataset2023",
    "IoT_23",
    "IoT_Network_Intrusion_Macro",
    "IoT_Network_Intrusion_Micro",
    "KDD_Cup_1999",
    "UNSW_NB15",
    "BCCC_CIC-BCCC-NRC-ACI-IOT-2023",
    "BCCC_CIC-BCCC-NRC-Edge-IIoTSet-2022",
    "BCCC_CIC-BCCC-NRC-IoMT-2024",
    "BCCC_CIC-BCCC-NRC-IoT-2022",
    "BCCC_CIC-BCCC-NRC-IoT-2023-Original_Training_and_Testing",
    "BCCC_CIC-BCCC-NRC-IoT-HCRL-2019",
    "BCCC_CIC-BCCC-NRC-MQTTIoT-IDS-2020",
    "BCCC_CIC-BCCC-NRC-TONIoT-2021",
    "BCCC_CIC-BCCC-NRC-UQ-IOT-2022",
    "BoT_IoT_Macro",
    "BoT_IoT_Micro",
    "CICAPT_IIoT_Phase1_Macro", # nok (single-class)
    "CICAPT_IIoT_Phase1_Micro", # nok (single-class)
    "CICAPT_IIoT_Phase2_Macro",
    "CICAPT_IIoT_Phase2_Micro",
    "CICEVSE2024_EVSE-A_Macro",
    "CICEVSE2024_EVSE-A_Micro",
    "CICEVSE2024_EVSE-B_Macro",
    "CICEVSE2024_EVSE-B_Micro",
    "CICIoMT2024_Bluetooth",
    "CICIoMT2024_WiFi_and_MQTT",
    "CICIoV2024_Decimal_Macro",
    "CICIoV2024_Decimal_Micro",
    "EDGE-IIOTSET_DNN-EdgeIIoT",
    "EDGE-IIOTSET_ML-EdgeIIoT",
    "MQTT_IoT_IDS2020_BiflowFeatures",
    "MQTT_IoT_IDS2020_PacketFeatures",
    "MQTT_IoT_IDS2020_UniflowFeatures",
    "NIDS_CIC-BoT-IoT",
    "NIDS_CIC-ToN-IoT",
    "NIDS_NF-BoT-IoT",
    "NIDS_NF-BoT-IoT-v2",
    "NIDS_NF-BoT-IoT-v3",
    "NIDS_NF-CICIDS2018-v3",
    "NIDS_NF-CSE-CIC-IDS2018",
    "NIDS_NF-CSE-CIC-IDS2018-v2",
    "NIDS_NF-ToN-IoT",
    "NIDS_NF-ToN-IoT-v2",
    "NIDS_NF-ToN-IoT-v3",
    "NIDS_NF-UNSW-NB15",
    "NIDS_NF-UNSW-NB15-v2",
    "NIDS_NF-UNSW-NB15-v3",
    "NIDS_NF-UQ-NIDS",
    "NIDS_NF-UQ-NIDS-v2",
    "N_BaIoT_Danmini_Doorbell",
    "N_BaIoT_Ecobee_Thermostat",
    "N_BaIoT_Ennio_Doorbell",
    "N_BaIoT_Philips_B120N10_Baby_Monitor",
    "N_BaIoT_Provision_PT_737E_Security_Camera",
    "N_BaIoT_Provision_PT_838_Security_Camera",
    "N_BaIoT_Samsung_SNH_1011_N_Webcam",
    "N_BaIoT_SimpleHome_XCS7_1002_WHT_Security_Camera",
    "N_BaIoT_SimpleHome_XCS7_1003_WHT_Security_Camera",
    "ToN_IoT_IoT_Fridge",
    "ToN_IoT_IoT_GPS_Tracker",
    "ToN_IoT_IoT_Garage_Door",
    "ToN_IoT_IoT_Modbus",
    "ToN_IoT_IoT_Motion_Light",
    "ToN_IoT_IoT_Thermostat",
    "ToN_IoT_IoT_Weather",
    "ToN_IoT_Linux_Disk",
    "ToN_IoT_Linux_Memory",
    "ToN_IoT_Linux_Process",
    "ToN_IoT_Network",
    "ToN_IoT_Windows_10",
    "ToN_IoT_Windows_7"
]

In [5]:
def blend_with_white(rgb, alpha):
    return [1 - alpha * (1 - c) for c in rgb]

def format_and_color_columns(df, color_map_dict={}, alpha=0.0):
    df_colored = df.copy()

    for col in df.columns:
        col_data = df[col]

        # === Step 1: Apply your custom formatting ===
        if pd.api.types.is_float_dtype(col_data):
            if 'time' in col:
                formatted = col_data.map(lambda x: f"{x:,.1f}")
            elif 'size' in col:
                formatted = col_data.map(lambda x: f"{x:,.2f}")
            else:
                formatted = col_data.map(lambda x: f"{x:,.3f}")
        elif pd.api.types.is_integer_dtype(col_data):
            formatted = col_data.map(lambda x: f"{x:,}")
        else:
            formatted = col_data.astype(str)

        # === Step 2: Apply LaTeX color using colormap if specified ===
        if col in color_map_dict and pd.api.types.is_numeric_dtype(col_data):
            cmap = colormaps[color_map_dict[col]]
            valid_mask = col_data.notna()
            norm = Normalize(vmin=col_data[valid_mask].min(), vmax=col_data[valid_mask].max())
    
            # Start with string-typed formatted column
            colored_column = formatted.astype(str).copy()
    
            # Compute blended RGB
            rgba_colors = cmap(norm(col_data[valid_mask]))[:, :3]
            blended_colors = [blend_with_white(rgb, alpha=alpha) for rgb in rgba_colors]
    
            for i, (r, g, b) in zip(col_data[valid_mask].index, blended_colors):
                df_colored.loc[i, col] = (
                    f"\\cellcolor[rgb]{{{r:.3f}, {g:.3f}, {b:.3f}}} {formatted[i]}"
                )
        else:
            df_colored[col] = formatted

    return df_colored

In [6]:
def flatten_metrics_dict(metrics_dict: dict, dataset_id: str, seed: int, keys_to_include=None) -> dict:
    flat = {"dataset_id": dataset_id, "seed": seed}
    for top_key, subdict in metrics_dict.items():
        if isinstance(subdict, dict):
            for sub_key, value in subdict.items():
                flat_key = f"{top_key}_{sub_key}"
                if keys_to_include is None or flat_key in keys_to_include:
                    flat[flat_key] = value
        else:
            if keys_to_include is None or top_key in keys_to_include:
                flat[top_key] = subdict
    return flat

def compute_composite_difficulty_from_dict(metrics_dict: dict, dataset_id: str, seed: int, min_metrics_per_group=1) -> pd.DataFrame | None:
    all_metrics = [m for group in metric_groups.values() for m in group]
    
    flat_dict = flatten_metrics_dict(metrics_dict, dataset_id, seed, keys_to_include=all_metrics)
    df = pd.DataFrame([flat_dict])
    df[df.select_dtypes(include=['float64']).columns] = df.select_dtypes(include=['float64']).astype('float32')
    df[df.select_dtypes(include=['int64']).columns] = df.select_dtypes(include=['int64']).astype('int32')
    
    available_metrics = []
    missing_metrics = []

    for metric in all_metrics:
        if metric in df.columns and not pd.isna(df.loc[0, metric]):
            available_metrics.append(metric)
        else:
            missing_metrics.append(metric)

    if missing_metrics:
        print(f"[INFO] {dataset_id}: Missing metrics: {missing_metrics}")
    
    valid_groups = {}
    for group_name, metric_list in metric_groups.items():
        valid_metrics_in_group = [m for m in metric_list if m in available_metrics]
        if len(valid_metrics_in_group) >= min_metrics_per_group:
            valid_groups[group_name] = valid_metrics_in_group
        else:
            print(f"[WARN] {dataset_id}: Group '{group_name}' has only {len(valid_metrics_in_group)} valid metrics")
    
    if len(valid_groups) < 3:
        print(f"[SKIP] {dataset_id}: Only {len(valid_groups)} valid groups, need at least 3")
        return None

    for metric in metrics_to_invert:
        if metric in available_metrics:
            df[metric] = -df[metric]

    # print(f"[INFO] {dataset_id}: Skipping normalization for single dataset")

    for group_name, metric_list in valid_groups.items():
        df[f"{group_name}_difficulty"] = df[metric_list].mean(axis=1)

    group_cols = [f"{g}_difficulty" for g in valid_groups.keys()]
    df["overall_difficulty"] = df[group_cols].mean(axis=1)
    df["metrics_used"] = len(available_metrics)
    df["groups_used"] = len(valid_groups)

    return df

def compute_all_composite_difficulties(root_dir: str, frac: str, suffix: str = ".complexity.json", min_metrics_per_group=1) -> pd.DataFrame:
    """
    Loads all complexity metric JSONs from a folder and computes composite difficulty scores.
    More flexible version that handles missing metrics gracefully.
    """
    
    exclude_substrings = {
        "CICAPT_IIoT_Phase1_Macro",
        "CICAPT_IIoT_Phase1_Micro",
        # "ToN_IoT_IoT_Motion_Light"
    }
    
    all_json_paths = [
        path for path in Path(root_dir).rglob(f"*{suffix}")
        if frac in str(path.resolve()) and not any(substr in path.stem for substr in exclude_substrings)
        # if 'full' in str(path.resolve()) and not any(substr in path.stem for substr in exclude_substrings)
    ]

    rows = []

    print(f"[{frac}]\tFound {len(all_json_paths)} JSON files to process")

    for path in tqdm(all_json_paths, desc="Computing composite difficulties"):
        try:
            seed = int(re.search(r'.*/seed_(\d+)/', str(path.resolve())).groups(1)[0])
            with open(path, 'r') as f:
                metrics_dict = json.load(f)

            # Remove metadata keys
            metrics_dict.pop('label_mappings', None)
            metrics_dict.pop('errors', None)

            dataset_id = path.stem.replace('_X_y.complexity', '')

            composite_df = compute_composite_difficulty_from_dict(
                metrics_dict, dataset_id, seed, min_metrics_per_group
            )
            # display(composite_df)
            
            if composite_df is not None:
                rows.append(composite_df)

        except Exception as e:
            print(f"[ERROR] Failed to process {path}: {e}")

    if not rows:
        print("No datasets could be processed")
        return pd.DataFrame()

    result_df = pd.concat(rows, ignore_index=True)
    # display(result_df)

    # Normalize across datasets
    print(f"\nNormalizing metrics across {len(result_df)} datasets...")

    # Determine metrics to normalize
    metric_cols = []
    for group_name, metric_list in metric_groups.items():
        metric_cols.extend(metric_list)

    available_metric_cols = [col for col in metric_cols if col in result_df.columns]

    # Fill NaNs before normalization if needed
    if result_df[available_metric_cols].isnull().values.any():
        print("[WARN] NaNs detected before normalization – filling with 0")
        result_df[available_metric_cols] = result_df[available_metric_cols].fillna(0)

    # Check for infinites
    inf_mask = np.isinf(result_df[available_metric_cols])
    if inf_mask.values.any():
        cols_with_inf = inf_mask.columns[inf_mask.any()].tolist()
        print(f"[ERROR] Infinity detected in columns: {cols_with_inf}")
        
        # Optional: See the max values to check for 'too large for float32'
        print("Max values per column:")
        print(result_df[available_metric_cols].max())
        return pd.DataFrame()

    # Normalize
    if available_metric_cols:
        scaler = MinMaxScaler()
        result_df[available_metric_cols] = scaler.fit_transform(result_df[available_metric_cols])

    # Recompute group difficulties
    for group_name, metric_list in metric_groups.items():
        available_group_metrics = [m for m in metric_list if m in result_df.columns]
        if available_group_metrics:
            result_df[f"{group_name}_difficulty"] = result_df[available_group_metrics].mean(axis=1)

    # Recompute overall difficulty
    group_cols = [f"{group_name}_difficulty" for group_name in metric_groups.keys()
                  if f"{group_name}_difficulty" in result_df.columns]
    if group_cols:
        result_df["overall_difficulty"] = result_df[group_cols].mean(axis=1)

    print(f"Successfully processed {len(result_df)} datasets")
    print(f"Average metrics used per dataset: {result_df['metrics_used'].mean():.1f}")
    print(f"Average groups used per dataset: {result_df['groups_used'].mean():.1f}")

    return result_df

In [7]:
ROOT_DIR = "../../2025-11-17/Output_Zip_v4_Complexity_v3"

SAMPLE_FRACS = ['sampled_05', 'sampled_10', 'sampled_20', 'sampled_25', 'sampled_50', 'full']

df_composite = {}

for frac in SAMPLE_FRACS:

    df_composite[frac] = compute_all_composite_difficulties(ROOT_DIR, frac, min_metrics_per_group=1)

    display(df_composite[frac])

[sampled_05]	Found 365 JSON files to process


Computing composite difficulties:  22%|██████████████████████████████████████▋                                                                                                                                            | 79/365 [00:00<00:01, 193.37it/s]

[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['pca_centroid_distance_pca_centroid_score', 'mahalanobis_class_distance_mean', 'svm_margin_mean', 'class_proba_entropy_mean', 'intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score', 'class_confusion_entropy_confusion_entropy']
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Local_Overlap' has only 0 valid metrics
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Boundary_Hardness' has only 0 valid metrics
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Global_Structure' has only 0 valid metrics
[SKIP] ToN_IoT_IoT_Motion_Light: Only 2 valid groups, need at least 3
[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['pca_centroid_distance_pca_centroid_score', 'mahalanobis_class_distance_mean', 'svm_margin_mean', 'class_proba_entropy_mean', 'intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score', 'class_confusion_entropy_confusion_entropy']
[WARN] ToN_IoT_IoT_Motion_L

Computing composite difficulties:  75%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                             | 272/365 [00:01<00:00, 184.82it/s]

[INFO] NIDS_NF-CICIDS2018-v3: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] NIDS_NF-CICIDS2018-v3: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] NIDS_NF-CICIDS2018-v3: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] NIDS_NF-CICIDS2018-v3: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] IoT_23: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] IoT_23: Missing metrics: ['mahalanobis_class_distance_mean']
[INFO] IoT_23: Missing metrics: ['mahalanobis_class_distance_mean']


...

Computing composite difficulties:  20%|███████████████████████████████████▊                                                                                                                                               | 73/365 [00:00<00:01, 157.83it/s]

[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score']
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Global_Structure' has only 0 valid metrics
[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score']
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Global_Structure' has only 0 valid metrics
[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score']
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Global_Structure' has only 0 valid metrics
[INFO] ToN_IoT_IoT_Motion_Light: Missing metrics: ['intrinsic_dimensionality_intrinsic_dimensionality_percent', 'calinski_harabasz_calinski_harabasz_score']
[WARN] ToN_IoT_IoT_Motion_Light: Group 'Global_Structure' has only 0 valid metrics
[INFO] ToN_IoT_IoT_Motion_Light: Missing

Computing composite difficulties: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 365/365 [00:02<00:00, 174.03it/s]



Normalizing metrics across 365 datasets...
[WARN] NaNs detected before normalization – filling with 0
Successfully processed 365 datasets
Average metrics used per dataset: 10.0
Average groups used per dataset: 5.0


,dataset_id,seed,anova_f_mean,mutual_info_mean,pca_centroid_distance_pca_centroid_score,mahalanobis_class_distance_mean,svm_margin_mean,class_proba_entropy_mean,intrinsic_dimensionality_intrinsic_dimensionality_percent,calinski_harabasz_calinski_harabasz_score,class_confusion_entropy_confusion_entropy,class_imbalance_normalized_entropy,Feature_Relevance_difficulty,Local_Overlap_difficulty,Boundary_Hardness_difficulty,Global_Structure_difficulty,Class_Distribution_Separation_difficulty,overall_difficulty,metrics_used,groups_used
0,BoT_IoT_Macro,53,9.990109e-01,0.905087,1.953808e-01,0.574324,0.930997,0.281326,0.400000,0.796860,5.599416e-01,0.474375,0.952049,0.384852,0.606162,0.598430,5.171582e-01,0.611730,10,5
1,BoT_IoT_Macro,23,9.990102e-01,0.904999,1.951659e-01,0.574119,0.931711,0.280740,0.400000,0.796543,5.564919e-01,0.474375,0.952005,0.384643,0.606225,0.598271,5.154333e-01,0.611315,10,5
2,BoT_IoT_Macro,37,9.989980e-01,0.904655,1.949853e-01,0.573588,0.932095,0.281924,0.400000,0.796452,5.669234e-01,0.474290,0.951827,0.384286,0.607009,0.598226,5.206066e-01,0.612391,10,5
3,BoT_IoT_Macro,89,9.990067e-01,0.904951,1.978069e-01,0.575638,0.930924,0.280681,0.400000,0.794893,5.480427e-01,0.474375,0.951979,0.386722,0.605802,0.597446,5.112087e-01,0.610632,10,5
4,BoT_IoT_Macro,17,9.990058e-01,0.904931,1.963975e-01,0.576306,0.931205,0.279941,0.400000,0.794453,5.526984e-01,0.474375,0.951968,0.386352,0.605573,0.597227,5.135365e-01,0.610931,10,5
5,NIDS_NF-ToN-IoT-v2,53,9.994627e-01,0.758122,9.090987e-02,0.411062,0.903763,0.324082,0.526316,0.942216,6.578479e-01,0.258118,0.878792,0.250986,0.613923,0.734266,4.579832e-01,0.587190,10,5
6,NIDS_NF-ToN-IoT-v2,23,9.994628e-01,0.758127,9.079023e-02,0.411278,0.903749,0.324711,0.526316,0.942270,6.605473e-01,0.258116,0.878795,0.251034,0.614230,0.734293,4.593317e-01,0.587537,10,5
7,NIDS_NF-ToN-IoT-v2,37,9.994629e-01,0.758177,9.086151e-02,0.411087,0.903705,0.324435,0.526316,0.942290,6.673835e-01,0.258113,0.878820,0.250974,0.614070,0.734303,4.627483e-01,0.588183,10,5
8,NIDS_NF-ToN-IoT-v2,89,9.994629e-01,0.758175,9.080595e-02,0.412872,0.903234,0.323039,0.526316,0.942289,6.619249e-01,0.258118,0.878819,0.251839,0.613136,0.734303,4.600216e-01,0.587624,10,5
9,NIDS_NF-ToN-IoT-v2,17,9.994629e-01,0.758164,9.090070e-02,0.412817,0.904169,0.323603,0.526316,0.942288,6.791548e-01,0.258118,0.878814,0.251859,0.613886,0.734302,4.686365e-01,0.589499,10,5


In [8]:
for frac in SAMPLE_FRACS:

    print(frac)
    try:
        df_composite[frac] = df_composite[frac].groupby('dataset_id', as_index=False).mean(numeric_only=True).drop(columns=['seed'])
    except Exception as e:
        print('error', e)
        continue

display(df_composite['full'])

sampled_05
error 'dataset_id'
sampled_10
sampled_20
sampled_25
sampled_50
full


,dataset_id,anova_f_mean,mutual_info_mean,pca_centroid_distance_pca_centroid_score,mahalanobis_class_distance_mean,svm_margin_mean,class_proba_entropy_mean,intrinsic_dimensionality_intrinsic_dimensionality_percent,calinski_harabasz_calinski_harabasz_score,class_confusion_entropy_confusion_entropy,class_imbalance_normalized_entropy,Feature_Relevance_difficulty,Local_Overlap_difficulty,Boundary_Hardness_difficulty,Global_Structure_difficulty,Class_Distribution_Separation_difficulty,overall_difficulty,metrics_used,groups_used
0,BCCC_CIC-BCCC-NRC-ACI-IOT-2023,9.999275e-01,0.804043,5.363762e-02,0.362441,0.925368,0.510467,0.273973,0.991203,5.952467e-01,0.237416,0.901985,0.208040,0.717917,0.632588,4.163311e-01,0.575372,10.0,5.0
1,BCCC_CIC-BCCC-NRC-Edge-IIoTSet-2022,9.958051e-01,0.782120,2.257247e-01,0.499741,0.712437,0.012031,0.173913,0.743210,2.978558e-01,0.752202,0.888962,0.362733,0.362234,0.458562,5.250289e-01,0.519504,10.0,5.0
2,BCCC_CIC-BCCC-NRC-IoMT-2024,9.997652e-01,0.682533,1.997377e-01,0.301290,0.833293,0.166076,0.219178,0.930076,7.373126e-01,0.452544,0.841149,0.250514,0.499685,0.574627,5.949281e-01,0.552181,10.0,5.0
3,BCCC_CIC-BCCC-NRC-IoT-2022,9.999865e-01,0.982795,9.218891e-02,0.301410,0.032206,0.003036,0.263889,0.998275,9.655223e-02,0.901102,0.991391,0.196799,0.017621,0.631082,4.988274e-01,0.467144,10.0,5.0
4,BCCC_CIC-BCCC-NRC-IoT-2023-Original_Training_a...,9.992814e-01,0.603465,1.102160e-01,0.416148,0.773053,0.055111,0.267606,0.925620,5.503181e-01,0.398317,0.801373,0.263182,0.414082,0.596613,4.743176e-01,0.509914,10.0,5.0
5,BCCC_CIC-BCCC-NRC-IoT-HCRL-2019,9.999248e-01,0.738776,1.860812e-01,0.565708,0.810843,0.138997,0.260274,0.984388,8.223315e-01,0.432346,0.869351,0.375894,0.474920,0.622331,6.273388e-01,0.593967,10.0,5.0
6,BCCC_CIC-BCCC-NRC-MQTTIoT-IDS-2020,9.879524e-01,0.170644,6.683876e-02,0.357698,0.701070,0.042604,0.101449,0.014240,4.779768e-01,0.440618,0.579298,0.212268,0.371837,0.057844,4.592977e-01,0.336109,10.0,5.0
7,BCCC_CIC-BCCC-NRC-TONIoT-2021,9.996886e-01,0.903233,8.204560e-02,0.312916,0.683617,0.053069,0.239437,0.948634,6.630396e-01,0.668412,0.951461,0.197481,0.368343,0.594035,6.657256e-01,0.555409,10.0,5.0
8,BCCC_CIC-BCCC-NRC-UQ-IOT-2022,9.989427e-01,0.907023,6.242871e-01,0.451664,0.737776,0.018467,0.202899,0.910938,4.125024e-01,0.466369,0.952983,0.537975,0.378122,0.556918,4.394355e-01,0.573087,10.0,5.0
9,BoT_IoT_Macro,9.990063e-01,0.904925,1.959473e-01,0.574795,0.931386,0.280922,0.400000,0.795840,5.568196e-01,0.474358,0.951965,0.385371,0.606154,0.597920,5.155886e-01,0.611400,10.0,5.0


In [9]:
df_composite_indexed = {}
df_ordered = {}

for frac in SAMPLE_FRACS:

    print(frac)

    try:
        
        # Set index to dataset_id
        df_composite_indexed[frac] = df_composite[frac].set_index('dataset_id')
        
        # Filter row_order to only include datasets that exist in the DataFrame
        valid_order = [d for d in row_order if d in df_composite_indexed[frac].index]
        
        # Reorder using the filtered list
        df_ordered[frac] = df_composite_indexed[frac].loc[valid_order]
        
        # Export to LaTeX with thousands separator
        df_ordered[frac].style.format(thousands=",").to_latex(f"tables/complexity_metrics_{frac}.tex")
        
        df_ordered[frac].to_excel(f'tables/complexity_metrics_{frac}.xlsx')
        df_ordered[frac].to_json(f'tables/complexity_metrics_{frac}.json', orient='index')

    except Exception as e:
        print('error', e)
        continue

sampled_05
error "None of ['dataset_id'] are in the columns"
sampled_10
sampled_20
sampled_25
sampled_50
full


In [10]:
df_ordered['full']

,anova_f_mean,mutual_info_mean,pca_centroid_distance_pca_centroid_score,mahalanobis_class_distance_mean,svm_margin_mean,class_proba_entropy_mean,intrinsic_dimensionality_intrinsic_dimensionality_percent,calinski_harabasz_calinski_harabasz_score,class_confusion_entropy_confusion_entropy,class_imbalance_normalized_entropy,Feature_Relevance_difficulty,Local_Overlap_difficulty,Boundary_Hardness_difficulty,Global_Structure_difficulty,Class_Distribution_Separation_difficulty,overall_difficulty,metrics_used,groups_used
dataset_id,,,,,,,,,,,,,,,,,,
CIC_IDS_2017,9.998266e-01,0.781049,1.912842e-01,0.171354,0.752435,0.013494,0.185714,0.963209,2.994197e-01,0.762430,0.890438,0.181319,0.382964,0.574462,5.309249e-01,0.512022,10.0,5.0
CIC_IOT_Dataset2023,9.956639e-01,0.251654,1.252598e-01,0.392455,0.928340,0.422491,0.333333,0.232210,7.643563e-01,0.143047,0.623659,0.258858,0.675416,0.282772,4.537015e-01,0.458881,10.0,5.0
IoT_23,9.997946e-01,0.811357,3.200495e-01,0.718751,0.800901,0.111346,0.500000,0.996208,5.767349e-01,0.764896,0.905576,0.519400,0.456124,0.748104,6.708153e-01,0.660004,10.0,5.0
IoT_Network_Intrusion_Macro,9.999914e-01,0.960318,3.919903e-02,0.576205,0.825956,0.177117,0.192308,0.998002,3.391479e-01,0.705840,0.980155,0.307702,0.501537,0.595155,5.224940e-01,0.581409,10.0,5.0
IoT_Network_Intrusion_Micro,9.999951e-01,0.955576,4.725233e-02,0.578219,0.831009,0.179887,0.192308,0.999066,1.983134e-01,0.795674,0.977786,0.312736,0.505448,0.595687,4.969937e-01,0.577730,10.0,5.0
KDD_Cup_1999,9.900512e-01,0.730730,1.031493e-01,0.548376,0.737175,0.008383,0.200000,0.844604,3.807039e-01,0.790417,0.860391,0.325763,0.372779,0.522302,5.855603e-01,0.533359,10.0,5.0
UNSW_NB15,9.994909e-01,0.937953,6.459107e-02,0.289099,0.666555,0.032260,0.200000,0.951991,9.973798e-01,0.885563,0.968722,0.176845,0.349407,0.575995,9.414715e-01,0.602488,10.0,5.0
BCCC_CIC-BCCC-NRC-ACI-IOT-2023,9.999275e-01,0.804043,5.363762e-02,0.362441,0.925368,0.510467,0.273973,0.991203,5.952467e-01,0.237416,0.901985,0.208040,0.717917,0.632588,4.163311e-01,0.575372,10.0,5.0
BCCC_CIC-BCCC-NRC-Edge-IIoTSet-2022,9.958051e-01,0.782120,2.257247e-01,0.499741,0.712437,0.012031,0.173913,0.743210,2.978558e-01,0.752202,0.888962,0.362733,0.362234,0.458562,5.250289e-01,0.519504,10.0,5.0


In [11]:
cols_to_drop = [
    "Feature_Relevance_difficulty", "Local_Overlap_difficulty", 
    "Boundary_Hardness_difficulty", "Global_Structure_difficulty", 
    "Class_Distribution_Separation_difficulty", 
    "metrics_used", "groups_used"
]

df_ordered_pretty = {}

for frac in SAMPLE_FRACS:

    try:
        df_ordered_pretty[frac] = format_and_color_columns(
            df_ordered[frac].drop(columns=cols_to_drop).loc[valid_order],
            color_map_dict={'overall_difficulty': 'RdYlGn_r'},
            alpha=0.5
        )
    
        with open(f"tables/table_2_{frac}.tex", "w") as f:
            f.write(df_ordered_pretty[frac].to_string())

    except Exception as e:
        print('error', e)
        continue
    
df_ordered_pretty['full']

error 'sampled_05'


/tmp/ipykernel_3612/3793402051.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[rgb]{0.966, 0.986, 0.829} 0.510' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df_colored.loc[i, col] = (
/tmp/ipykernel_3612/3793402051.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[rgb]{0.955, 0.981, 0.813} 0.514' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df_colored.loc[i, col] = (
/tmp/ipykernel_3612/3793402051.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '\cellcolor[rgb]{0.978, 0.991, 0.845} 0.517' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df_colored.loc[i, col] = (
/tmp/ipykerne

,anova_f_mean,mutual_info_mean,pca_centroid_distance_pca_centroid_score,mahalanobis_class_distance_mean,svm_margin_mean,class_proba_entropy_mean,intrinsic_dimensionality_intrinsic_dimensionality_percent,calinski_harabasz_calinski_harabasz_score,class_confusion_entropy_confusion_entropy,class_imbalance_normalized_entropy,overall_difficulty
dataset_id,,,,,,,,,,,
CIC_IDS_2017,1.000,0.781,0.191,0.171,0.752,0.013,0.186,0.963,0.299,0.762,"\cellcolor[rgb]{0.999, 0.973, 0.829} 0.512"
CIC_IOT_Dataset2023,0.996,0.252,0.125,0.392,0.928,0.422,0.333,0.232,0.764,0.143,"\cellcolor[rgb]{0.906, 0.960, 0.760} 0.459"
IoT_23,1.000,0.811,0.320,0.719,0.801,0.111,0.500,0.996,0.577,0.765,"\cellcolor[rgb]{0.824, 0.500, 0.575} 0.660"
IoT_Network_Intrusion_Macro,1.000,0.960,0.039,0.576,0.826,0.177,0.192,0.998,0.339,0.706,"\cellcolor[rgb]{0.986, 0.769, 0.657} 0.581"
IoT_Network_Intrusion_Micro,1.000,0.956,0.047,0.578,0.831,0.180,0.192,0.999,0.198,0.796,"\cellcolor[rgb]{0.988, 0.784, 0.664} 0.578"
KDD_Cup_1999,0.990,0.731,0.103,0.548,0.737,0.008,0.200,0.845,0.381,0.790,"\cellcolor[rgb]{0.998, 0.932, 0.766} 0.533"
UNSW_NB15,0.999,0.938,0.065,0.289,0.667,0.032,0.200,0.952,0.997,0.886,"\cellcolor[rgb]{0.965, 0.686, 0.618} 0.602"
BCCC_CIC-BCCC-NRC-ACI-IOT-2023,1.000,0.804,0.054,0.362,0.925,0.510,0.274,0.991,0.595,0.237,"\cellcolor[rgb]{0.989, 0.789, 0.666} 0.575"
BCCC_CIC-BCCC-NRC-Edge-IIoTSet-2022,0.996,0.782,0.226,0.500,0.712,0.012,0.174,0.743,0.298,0.752,"\cellcolor[rgb]{0.999, 0.961, 0.809} 0.520"


In [12]:
styled_df = {}

for frac in SAMPLE_FRACS:

    print(frac)

    try:

        # Compute min and max of overall_difficulty
        vmin = df_ordered[frac]['overall_difficulty'].min()
        vmax = df_ordered[frac]['overall_difficulty'].max()
        
        # Select and style (don't include 'dataset_id' as it's now the index)
        styled_df[frac] = df_ordered[frac][['overall_difficulty', 'metrics_used', 'groups_used']].style \
            .background_gradient(subset=['overall_difficulty'], cmap='RdYlGn_r', vmin=vmin, vmax=vmax) \
            .format({'overall_difficulty': '{:.3f}'})

    except Exception as e:
        print('error', e)
    
styled_df['full']

sampled_05
error 'sampled_05'
sampled_10
sampled_20
sampled_25
sampled_50
full


,overall_difficulty,metrics_used,groups_used
dataset_id,,,
CIC_IDS_2017,0.512,10.000000,5.000000
CIC_IOT_Dataset2023,0.459,10.000000,5.000000
IoT_23,0.660,10.000000,5.000000
IoT_Network_Intrusion_Macro,0.581,10.000000,5.000000
IoT_Network_Intrusion_Micro,0.578,10.000000,5.000000
KDD_Cup_1999,0.533,10.000000,5.000000
UNSW_NB15,0.602,10.000000,5.000000
BCCC_CIC-BCCC-NRC-ACI-IOT-2023,0.575,10.000000,5.000000
BCCC_CIC-BCCC-NRC-Edge-IIoTSet-2022,0.520,10.000000,5.000000
